# Job Postings Dataset — EDA & Salary Prediction

**15,000 synthetic job listings | 10 titles | 8 industries | 2023–2025**

---

> **TL;DR** — This notebook provides a thorough exploration of a synthetic job postings dataset built for NLP and salary prediction tasks. We analyze salary distributions, remote work trends, skills demand, and industry pay gaps. We then train two baseline ML models: a **TF-IDF + Logistic Regression classifier** to predict job title from description, and an **XGBoost regressor** to predict `salary_min` from structured features.

**Contents:**
1. [Setup & Data Loading](#1)
2. [Dataset Overview](#2)
3. [Job Title Distribution](#3)
4. [Salary Analysis by Experience & Title](#4)
5. [Remote Work Trends](#5)
6. [Skills Analysis](#6)
7. [Salary by Company Size](#7)
8. [Industry Analysis](#8)
9. [Correlation Matrix](#9)
10. [Skill Count vs Salary](#10)
11. [NLP Baseline: Title Classification](#11)
12. [Salary Prediction Baseline (XGBoost)](#12)
13. [ML Use Cases](#13)
14. [Conclusion](#14)

---

If you find this exploration useful, please **upvote the dataset and this notebook**!

<a id='1'></a>
## 1. Setup & Data Loading

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.figsize'] = (12, 5)
matplotlib.rcParams['font.size'] = 11
plt.style.use('seaborn-v0_8-whitegrid')

# Load data — supports both Kaggle kernel path and local path
kaggle_path = '/kaggle/input/job-postings-nlp-salary-prediction/job_postings.csv'
local_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'job_postings.csv')

if os.path.exists(kaggle_path):
    df = pd.read_csv(kaggle_path)
    print('Loaded from Kaggle input directory.')
elif os.path.exists('job_postings.csv'):
    df = pd.read_csv('job_postings.csv')
    print('Loaded from local directory.')
else:
    raise FileNotFoundError(
        'job_postings.csv not found. Run create_dataset.py first, '
        'or verify the Kaggle dataset path.'
    )

df['posted_date'] = pd.to_datetime(df['posted_date'])
df['year'] = df['posted_date'].dt.year
print(f'Dataset shape: {df.shape}')

<a id='2'></a>
## 2. Dataset Overview

In [ ]:
print(f'Shape : {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Date range: {df["posted_date"].min().date()} to {df["posted_date"].max().date()}')
print(f'Missing values: {df.isnull().sum().sum()}')
print()
print('Column dtypes:')
print(df.dtypes)
print()
df.head()

In [ ]:
# Numeric summary
df[['salary_min', 'salary_max', 'applications', 'days_to_fill']].describe().round(0)

<a id='3'></a>
## 3. Job Title Distribution

In [ ]:
title_counts = df['title'].value_counts()

fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.cm.Set2(np.linspace(0, 1, len(title_counts)))
bars = ax.bar(title_counts.index, title_counts.values, color=colors, edgecolor='white', linewidth=0.8)
ax.set_title('Job Posting Count by Title', fontsize=14, fontweight='bold')
ax.set_xlabel('Job Title')
ax.set_ylabel('Number of Postings')
ax.tick_params(axis='x', rotation=35)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print('Title distribution:')
print(title_counts.to_frame().rename(columns={'title': 'count'}))

<a id='4'></a>
## 4. Salary Analysis by Experience Level & Job Title

In [ ]:
exp_order = ['entry', 'mid', 'senior', 'lead', 'director']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot: salary_min by experience_level
data_by_exp = [df[df['experience_level'] == lvl]['salary_min'].values for lvl in exp_order]
bp1 = axes[0].boxplot(data_by_exp, labels=exp_order, patch_artist=True,
                       medianprops=dict(color='black', linewidth=2))
colors_exp = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c', '#9b59b6']
for patch, color in zip(bp1['boxes'], colors_exp):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_title('Salary Min by Experience Level', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Experience Level')
axes[0].set_ylabel('Salary Min (USD)')
axes[0].yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Box plot: salary_min by title
title_order = df.groupby('title')['salary_min'].median().sort_values(ascending=False).index.tolist()
data_by_title = [df[df['title'] == t]['salary_min'].values for t in title_order]
bp2 = axes[1].boxplot(data_by_title, labels=title_order, patch_artist=True,
                       medianprops=dict(color='black', linewidth=2), vert=False)
title_colors = plt.cm.Set3(np.linspace(0, 1, len(title_order)))
for patch, color in zip(bp2['boxes'], title_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[1].set_title('Salary Min by Job Title', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Salary Min (USD)')
axes[1].xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.show()

print('Median salary_min by experience level:')
print(df.groupby('experience_level')['salary_min'].median().reindex(exp_order).apply(lambda x: f'${x:,.0f}'))

<a id='5'></a>
## 5. Remote Work Trends

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall pie chart
remote_counts = df['remote_type'].value_counts()
pie_colors = ['#3498db', '#e67e22', '#2ecc71']
axes[0].pie(
    remote_counts.values,
    labels=remote_counts.index,
    autopct='%1.1f%%',
    colors=pie_colors,
    startangle=90,
    textprops={'fontsize': 12}
)
axes[0].set_title('Remote Work Type — Overall', fontsize=13, fontweight='bold')

# Stacked bar by year
remote_by_year = df.groupby(['year', 'remote_type']).size().unstack(fill_value=0)
remote_pct_by_year = remote_by_year.div(remote_by_year.sum(axis=1), axis=0) * 100
remote_pct_by_year[['onsite', 'hybrid', 'remote']].plot(
    kind='bar', stacked=True, ax=axes[1],
    color=['#3498db', '#e67e22', '#2ecc71'],
    edgecolor='white'
)
axes[1].set_title('Remote Type Mix by Year (%)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Percentage of Postings')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Remote Type', loc='upper right')

plt.tight_layout()
plt.show()

print('Remote type counts:')
print(remote_counts)
print()
print('Remote type by year (%):')
print(remote_pct_by_year.round(1))

<a id='6'></a>
## 6. Skills Analysis

In [ ]:
# Parse pipe-separated skills
all_skills = []
for row in df['required_skills']:
    all_skills.extend(str(row).split('|'))

skill_counts = Counter(all_skills)
top20 = pd.Series(dict(skill_counts.most_common(20))).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(11, 8))
bars = ax.barh(top20.index, top20.values, color='steelblue', edgecolor='white')
ax.set_title('Top 20 Most Required Skills (all roles)', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Job Postings Requiring Skill')
for bar in bars:
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height() / 2,
            f'{int(bar.get_width()):,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Skills by job title heatmap
# Pick top 15 skills overall
top15_skills = [s for s, _ in skill_counts.most_common(15)]
titles_list = df['title'].unique()

heatmap_data = pd.DataFrame(index=titles_list, columns=top15_skills, dtype=float)
for title in titles_list:
    subset = df[df['title'] == title]['required_skills']
    title_skills = []
    for row in subset:
        title_skills.extend(str(row).split('|'))
    total = len(subset)
    for skill in top15_skills:
        heatmap_data.loc[title, skill] = title_skills.count(skill) / total * 100

heatmap_data = heatmap_data.astype(float)

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(
    heatmap_data,
    annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.5,
    cbar_kws={'label': '% of postings requiring skill'},
    ax=ax
)
ax.set_title('Skill Frequency by Job Title (% of postings)', fontsize=14, fontweight='bold')
ax.set_ylabel('Job Title')
ax.set_xlabel('Skill')
plt.tight_layout()
plt.show()

<a id='7'></a>
## 7. Salary by Company Size

In [ ]:
size_order = ['startup', 'small', 'medium', 'large', 'enterprise']
palette = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db']

fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(
    data=df,
    x='company_size', y='salary_min',
    order=size_order,
    palette=palette,
    inner='quartile',
    ax=ax
)
ax.set_title('Salary Min Distribution by Company Size', fontsize=14, fontweight='bold')
ax.set_xlabel('Company Size')
ax.set_ylabel('Salary Min (USD)')
ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
plt.tight_layout()
plt.show()

print('Median salary_min by company size:')
medians = df.groupby('company_size')['salary_min'].median().reindex(size_order)
print(medians.apply(lambda x: f'${x:,.0f}'))

<a id='8'></a>
## 8. Industry Analysis — Median Salary

In [ ]:
industry_salary = df.groupby('industry')['salary_min'].median().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Median salary by industry
colors_ind = plt.cm.coolwarm(np.linspace(0.2, 0.8, len(industry_salary)))
bars = axes[0].bar(industry_salary.index, industry_salary.values, color=colors_ind, edgecolor='white')
axes[0].set_title('Median Salary Min by Industry', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Industry')
axes[0].set_ylabel('Median Salary Min (USD)')
axes[0].tick_params(axis='x', rotation=35)
axes[0].yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
                 f'${bar.get_height()/1000:.0f}K', ha='center', va='bottom', fontsize=9)

# Posting count by industry
industry_counts = df['industry'].value_counts()
axes[1].bar(industry_counts.index, industry_counts.values,
            color='steelblue', edgecolor='white')
axes[1].set_title('Number of Postings by Industry', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Industry')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()

<a id='9'></a>
## 9. Correlation Matrix

In [ ]:
# Encode ordinal variables for correlation analysis
exp_map = {'entry': 1, 'mid': 2, 'senior': 3, 'lead': 4, 'director': 5}
size_map = {'startup': 1, 'small': 2, 'medium': 3, 'large': 4, 'enterprise': 5}
remote_map = {'onsite': 0, 'hybrid': 1, 'remote': 2}

corr_df = df[['salary_min', 'salary_max', 'applications', 'days_to_fill']].copy()
corr_df['experience_num'] = df['experience_level'].map(exp_map)
corr_df['company_size_num'] = df['company_size'].map(size_map)
corr_df['remote_num'] = df['remote_type'].map(remote_map)
corr_df['skill_count'] = df['required_skills'].apply(lambda x: len(str(x).split('|')))

corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True, fmt='.2f',
    cmap='RdBu_r', center=0,
    linewidths=0.5,
    square=True,
    cbar_kws={'shrink': 0.8},
    ax=ax
)
ax.set_title('Correlation Matrix (Numeric & Encoded Ordinal Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='10'></a>
## 10. Skill Count vs Salary

In [ ]:
df['skill_count'] = df['required_skills'].apply(lambda x: len(str(x).split('|')))

# Mean salary by skill count
skill_salary = df.groupby('skill_count')['salary_min'].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Scatter: individual postings (sample 2000 for clarity)
sample = df.sample(min(2000, len(df)), random_state=42)
scatter = axes[0].scatter(
    sample['skill_count'], sample['salary_min'],
    c=sample['skill_count'], cmap='viridis', alpha=0.4, s=20
)
plt.colorbar(scatter, ax=axes[0], label='Skill Count')
axes[0].set_title('Skill Count vs Salary Min (sample n=2,000)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Number of Required Skills')
axes[0].set_ylabel('Salary Min (USD)')
axes[0].yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# Bar: mean salary by skill count
axes[1].bar(skill_salary['skill_count'], skill_salary['salary_min'],
            color='teal', edgecolor='white')
axes[1].set_title('Mean Salary Min by Skill Count', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of Required Skills')
axes[1].set_ylabel('Mean Salary Min (USD)')
axes[1].yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.show()

corr_val = df['skill_count'].corr(df['salary_min'])
print(f'Pearson correlation (skill_count, salary_min): {corr_val:.3f}')

<a id='11'></a>
## 11. NLP Baseline: Job Title Classification from Description

We use TF-IDF features on the `description` column and train a Logistic Regression to predict the `title` label. This is a direct measure of how much information the description text carries about the role.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

X_text = df['description'].fillna('')
y_title = df['title']

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_text, y_title, test_size=0.2, random_state=42, stratify=y_title
)

nlp_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)),
    ('clf', LogisticRegression(max_iter=1000, C=1.0, random_state=42)),
])

nlp_pipeline.fit(X_train_t, y_train_t)
y_pred_t = nlp_pipeline.predict(X_test_t)

acc = accuracy_score(y_test_t, y_pred_t)
print(f'TF-IDF + Logistic Regression')
print(f'Test Accuracy: {acc:.4f} ({acc*100:.1f}%)')
print()
print(classification_report(y_test_t, y_pred_t))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

cm = confusion_matrix(y_test_t, y_pred_t, labels=sorted(y_title.unique()))
fig, ax = plt.subplots(figsize=(11, 9))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y_title.unique()))
disp.plot(ax=ax, colorbar=True, cmap='Blues', xticks_rotation=45)
ax.set_title(f'Confusion Matrix — TF-IDF + LR (Accuracy: {acc:.1%})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='12'></a>
## 12. Salary Prediction Baseline (XGBoost)

We encode all categorical features, run a 5-fold cross-validated XGBoost regressor predicting `salary_min`, and visualize feature importance.

In [ ]:
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    from sklearn.ensemble import GradientBoostingRegressor
    HAS_XGB = False
    print('XGBoost not available — falling back to sklearn GradientBoostingRegressor.')

from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

# Feature engineering
reg_df = df.copy()
reg_df['skill_count'] = reg_df['required_skills'].apply(lambda x: len(str(x).split('|')))

cat_cols = ['title', 'experience_level', 'company_size', 'remote_type', 'industry', 'education_required']
for col in cat_cols:
    le = LabelEncoder()
    reg_df[col + '_enc'] = le.fit_transform(reg_df[col].astype(str))

feature_cols = [c + '_enc' for c in cat_cols] + ['skill_count', 'applications', 'days_to_fill']
target_col = 'salary_min'

X_reg = reg_df[feature_cols]
y_reg = reg_df[target_col]

if HAS_XGB:
    model = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0)
else:
    model = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_scores = np.sqrt(-cross_val_score(model, X_reg, y_reg, cv=kf,
                                        scoring='neg_mean_squared_error'))
r2_scores = cross_val_score(model, X_reg, y_reg, cv=kf, scoring='r2')

print('XGBoost Salary Prediction — 5-Fold Cross-Validation')
print(f'  RMSE : ${rmse_scores.mean():>10,.0f} (+/- ${rmse_scores.std():,.0f})')
print(f'  R²   : {r2_scores.mean():>10.4f} (+/- {r2_scores.std():.4f})')

In [ ]:
# Train on full data and plot feature importance
model.fit(X_reg, y_reg)

feat_importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=True)
readable_names = {
    'title_enc': 'Job Title',
    'experience_level_enc': 'Experience Level',
    'company_size_enc': 'Company Size',
    'remote_type_enc': 'Remote Type',
    'industry_enc': 'Industry',
    'education_required_enc': 'Education Required',
    'skill_count': 'Skill Count',
    'applications': 'Applications',
    'days_to_fill': 'Days to Fill',
}
feat_importance.index = [readable_names.get(f, f) for f in feat_importance.index]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c' if v == feat_importance.max() else 'steelblue' for v in feat_importance.values]
feat_importance.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('XGBoost Feature Importance — Salary Prediction', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Feature importances (sorted):')
print(feat_importance.sort_values(ascending=False).round(4))

<a id='13'></a>
## 13. ML Use Cases

| Project | Complexity | Target Column | Suggested Approach |
|---------|------------|---------------|--------------------|
| Salary Prediction (Regression) | Beginner | `salary_min` / `salary_max` | XGBoost, Ridge Regression |
| Job Title Classification (NLP) | Beginner | `title` | TF-IDF + LR, fine-tuned BERT |
| Skills Extraction | Intermediate | `required_skills` | NER, keyword extraction, rule-based parsing |
| Remote Work Prediction | Intermediate | `remote_type` | Multiclass classification on text + tabular |
| Demand Modeling | Intermediate | `applications` | Poisson regression, count models |
| Time-to-Fill Regression | Intermediate | `days_to_fill` | Survival analysis, gradient boosting |
| Company Size Inference | Advanced | `company_size` | Multi-modal (text + structured) |
| Salary Range Estimation from Description | Advanced | `salary_min`, `salary_max` | Seq2Seq, NLP regression |
| Skills Co-occurrence Graph | Advanced | `required_skills` | Graph analysis, community detection |
| Transfer Learning Benchmark | Advanced | `title` | BERT, RoBERTa, DistilBERT fine-tuning |

<a id='14'></a>
## 14. Conclusion

### Key Findings

1. **Salary scales strongly with experience**: Director-level roles earn 2-3x entry-level salaries in the same industry and company size.
2. **Enterprise pays a clear premium**: Enterprise companies pay ~20-35% more than startups for equivalent roles, visible in the violin plots.
3. **ML and Data Science roles command a salary premium**: `ML Engineer` and `Research Scientist` sit at the top of the salary distribution across experience levels.
4. **Remote work carries a salary bump**: Remote postings show ~10% higher median salary vs. onsite, reflecting geographic premium and talent competition.
5. **Finance and Healthcare are the highest-paying industries**: Media and Education trail significantly, consistent with real-world patterns.
6. **NLP baseline achieves strong title classification**: TF-IDF + Logistic Regression achieves high accuracy because description templates differ meaningfully across roles.
7. **Experience level is the single most important feature** for salary prediction — it alone explains the majority of variance.

### Next Steps

- Fine-tune a pre-trained language model (DistilBERT) on the `description` → `title` classification task.
- Build a salary range estimator using the description text alone (NLP regression).
- Analyze skills co-occurrence as a graph to identify skill clusters and career paths.
- Apply survival analysis to model `days_to_fill` as a time-to-event problem.

---

**Dataset and notebook by Lorenzo Scaturchio.**

### If you found this exploration useful, please upvote both the dataset and this notebook — it helps the community discover quality resources!